# 04 — Chronological evaluation

Evaluates generalisation to later observations using a timestamp-ordered 60/20/20 split. This is the primary deployment-oriented evaluation protocol.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
sys.path.append(str(Path.cwd().parent / 'src'))
from data import load_dataset, FEATURES, TARGET
from splits import chronological_60_20_20
from models import canonical_models
from evaluation import classification_metrics, select_threshold_by_validation_f1

df = load_dataset(Path('../data/raw/iotdata.csv'))
train, validation, test = chronological_60_20_20(df)
X_train, y_train = train[FEATURES], train[TARGET]
X_val, y_val = validation[FEATURES], validation[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
print(len(train), len(validation), len(test))
print(train[TARGET].sum(), validation[TARGET].sum(), test[TARGET].sum())

In [ ]:
results = []
for name, model in canonical_models().items():
    model.fit(X_train, y_train)
    val_score = model.predict_proba(X_val)[:, 1]
    threshold, _ = select_threshold_by_validation_f1(y_val, val_score)
    test_score = model.predict_proba(X_test)[:, 1]
    results.append({'model': name, **classification_metrics(y_test, test_score, threshold)})
pd.DataFrame(results)

## Canonical temporal reference

The audited record reports AP 0.004150 for Logistic Regression, 0.002801 for Random Forest and 0.004028 for HistGradientBoosting. The temporal test contains 92 positive events, so recall uncertainty is material.